In [1]:
import json
import os
import random
import duckdb
import pandas as pd
from datetime import datetime, timedelta

# Connect to DuckDB
con = duckdb.connect()

# Path to Synthea output
FHIR_PATH = os.path.expanduser("~/synthea/output/fhir")

print(f"DuckDB version: {duckdb.__version__}")
print(f"FHIR path exists: {os.path.exists(FHIR_PATH)}")
print(f"Patient files found: {len([f for f in os.listdir(FHIR_PATH) if f.endswith('.json') and 'Information' not in f])}")

DuckDB version: 1.5.0
FHIR path exists: True
Patient files found: 142


In [2]:
# Drop and recreate tables fresh each run
con.execute("DROP TABLE IF EXISTS bp_observations")
con.execute("DROP TABLE IF EXISTS conditions")
con.execute("DROP TABLE IF EXISTS patients")

# Patient demographics
con.execute("""
    CREATE TABLE patients (
        patient_id VARCHAR,
        name VARCHAR,
        birth_date DATE,
        age INTEGER,
        gender VARCHAR,
        deceased BOOLEAN
    )
""")

# Conditions including HTN, exclusions, and synthetic COVID
con.execute("""
    CREATE TABLE conditions (
        patient_id VARCHAR,
        condition_display VARCHAR,
        condition_code VARCHAR,
        onset_date DATE,
        is_htn BOOLEAN,
        is_covid BOOLEAN,
        is_esrd BOOLEAN,
        is_hospice BOOLEAN
    )
""")

# Blood pressure observations
con.execute("""
    CREATE TABLE bp_observations (
        patient_id VARCHAR,
        observation_date DATE,
        systolic INTEGER,
        diastolic INTEGER
    )
""")

print("Tables created: patients, conditions, bp_observations")

Tables created: patients, conditions, bp_observations


In [3]:
# Parse all Synthea FHIR JSON files into DuckDB tables
random.seed(42)  # Reproducible results

patients_data = []
conditions_data = []
bp_data = []

files = [f for f in os.listdir(FHIR_PATH) if f.endswith('.json') and 'Information' not in f]

for filename in files:
    filepath = os.path.join(FHIR_PATH, filename)
    
    with open(filepath) as f:
        bundle = json.load(f)
    
    patient_id = None
    birth_date = None
    
    for entry in bundle['entry']:
        resource = entry['resource']
        rtype = resource['resourceType']
        
        # --- Patient ---
        if rtype == 'Patient':
            patient_id = resource['id']
            name = resource['name'][0]
            full_name = ' '.join(name.get('given', [])) + ' ' + name.get('family', '')
            birth_date = datetime.strptime(resource['birthDate'], '%Y-%m-%d').date()
            age = (datetime.today().date() - birth_date).days // 365
            gender = resource.get('gender', 'unknown')
            deceased = 'deceasedDateTime' in resource or resource.get('deceasedBoolean', False)
            patients_data.append((patient_id, full_name.strip(), birth_date, age, gender, deceased))
        
        # --- Conditions ---
        elif rtype == 'Condition' and patient_id:
            code_block = resource.get('code', {})
            onset = resource.get('onsetDateTime', resource.get('onsetPeriod', {}).get('start'))
            onset_date = datetime.strptime(onset[:10], '%Y-%m-%d').date() if onset else None
            
            for coding in code_block.get('coding', []):
                display = coding.get('display', '')
                code = coding.get('code', '')
                
                is_htn = 'hypertension' in display.lower()
                is_esrd = any(x in display.lower() for x in ['renal failure', 'esrd', 'end-stage renal'])
                is_hospice = 'hospice' in display.lower()
                is_covid = False  # Will assign synthetically below
                
                conditions_data.append((
                    patient_id, display, code, onset_date,
                    is_htn, is_covid, is_esrd, is_hospice
                ))
        
        # --- Blood Pressure Observations ---
        elif rtype == 'Observation' and patient_id:
            codings = resource.get('code', {}).get('coding', [])
            is_bp_panel = any('85354-9' in c.get('code', '') for c in codings)  # Fixed LOINC code
            
            if is_bp_panel:
                obs_date_str = resource.get('effectiveDateTime', '')
                obs_date = datetime.strptime(obs_date_str[:10], '%Y-%m-%d').date() if obs_date_str else None
                
                systolic = None
                diastolic = None
                
                for component in resource.get('component', []):
                    comp_code = component.get('code', {}).get('coding', [{}])[0].get('code', '')
                    value = component.get('valueQuantity', {}).get('value')
                    if comp_code == '8480-6':  # Systolic
                        systolic = int(value) if value else None
                    elif comp_code == '8462-4':  # Diastolic
                        diastolic = int(value) if value else None
                
                if systolic and diastolic and obs_date:
                    bp_data.append((patient_id, obs_date, systolic, diastolic))

# Insert into DuckDB
con.executemany("INSERT INTO patients VALUES (?, ?, ?, ?, ?, ?)", patients_data)
con.executemany("INSERT INTO conditions VALUES (?, ?, ?, ?, ?, ?, ?, ?)", conditions_data)
con.executemany("INSERT INTO bp_observations VALUES (?, ?, ?, ?)", bp_data)

print("Patients loaded:      ", len(patients_data))
print("Condition records:    ", len(conditions_data))
print("BP observations:      ", len(bp_data))

htn_count = con.execute("SELECT COUNT(DISTINCT patient_id) FROM conditions WHERE is_htn = true").fetchone()[0]
bp_count = con.execute("SELECT COUNT(DISTINCT patient_id) FROM bp_observations").fetchone()[0]

print("Patients with HTN:    ", htn_count)
print("Patients with BP data:", bp_count)

Patients loaded:       142
Condition records:     6640
BP observations:       2749
Patients with HTN:     63
Patients with BP data: 142


In [4]:
# Assign synthetic COVID-19 diagnosis dates to ~30% of HTN patients
# Range reflects ongoing transmission well beyond PHE declaration end
# 3rd+ infections associated with long COVID, new-onset HTN, and POTS
random.seed(42)

covid_start = datetime(2020, 3, 1).date()
covid_end = datetime(2024, 12, 31).date()
covid_range_days = (covid_end - covid_start).days

# Get HTN patient IDs
htn_patients = con.execute(
    "SELECT DISTINCT patient_id FROM conditions WHERE is_htn = true"
).fetchall()
htn_ids = [row[0] for row in htn_patients]

# Randomly select 30%
covid_patients = random.sample(htn_ids, k=int(len(htn_ids) * 0.30))

# Insert synthetic COVID condition rows
covid_rows = []
for pid in covid_patients:
    covid_date = covid_start + timedelta(days=random.randint(0, covid_range_days))
    covid_rows.append((pid, 'COVID-19 (disorder)', 'U07.1', covid_date, False, True, False, False))

con.executemany("INSERT INTO conditions VALUES (?, ?, ?, ?, ?, ?, ?, ?)", covid_rows)

print("HTN patients:             ", len(htn_ids))
print("Assigned COVID diagnosis: ", len(covid_patients))
print("COVID date range:          2020-03-01 to 2024-12-31")

HTN patients:              63
Assigned COVID diagnosis:  18
COVID date range:          2020-03-01 to 2024-12-31


In [5]:
# HEDIS CBP Denominator
# Include: HTN patients, 18-85, not deceased
# Exclude: ESRD, hospice

denominator = con.execute("""
    SELECT DISTINCT
        p.patient_id,
        p.name,
        p.age,
        p.gender
    FROM patients p
    JOIN conditions c ON p.patient_id = c.patient_id
    WHERE c.is_htn = true
      AND p.age BETWEEN 18 AND 85
      AND p.deceased = false
      AND p.patient_id NOT IN (
          SELECT DISTINCT patient_id FROM conditions WHERE is_esrd = true
      )
      AND p.patient_id NOT IN (
          SELECT DISTINCT patient_id FROM conditions WHERE is_hospice = true
      )
    ORDER BY p.age
""").df()

print("Denominator (eligible HTN patients):", len(denominator))
print()
print(denominator.to_string(index=False))

Denominator (eligible HTN patients): 46

                          patient_id                                name  age gender
bd7193ce-d4f6-ff4c-4e96-f5c38b326484          Leona665 Sau887 Nicolas769   55 female
dd4f2c1c-087f-e859-11ee-b2221b0f92f2       Irwin931 Claud279 Armstrong51   55   male
3510ff85-54dc-c374-99b0-ec96fb92b0b6           Corinna386 Suk497 West559   55 female
c3f70f63-9cb5-ecf6-835b-f71e748acf73             Song837 Tena12 Hauck852   57 female
4ccd6636-2c8a-c36f-90b3-78ebb75c75f1          Nolan344 Kevin729 Lynch190   57   male
1474d404-d8e5-d961-4a0c-e1672d099f81       Dorian295 Nathan164 Rippin620   57   male
68aa3fc5-86c1-ef11-59bf-665c57c23556      Elisha578 Jack927 Wilkinson796   59   male
8b188bb8-e4b8-2993-89cf-3c893041acac       Salena230 Nenita289 Beahan375   59 female
13ee78eb-a477-1b7c-12a9-2baf8eead314        Anton902 Bryan958 Strosin214   59   male
67f26e88-a002-070f-73e2-e903e44daa2b             Creola518 Wintheiser220   60 female
95dddd60-1bca-9f56-0857-

In [6]:
# HEDIS CBP Numerator
# Use most recent BP reading per patient (per NCQA spec)
# Controlled = Systolic < 140 AND Diastolic < 90

numerator = con.execute("""
    WITH most_recent_bp AS (
        SELECT
            patient_id,
            systolic,
            diastolic,
            observation_date,
            ROW_NUMBER() OVER (
                PARTITION BY patient_id
                ORDER BY observation_date DESC
            ) AS rn
        FROM bp_observations
    )
    SELECT
        p.patient_id,
        p.name,
        p.age,
        p.gender,
        b.observation_date AS bp_date,
        b.systolic,
        b.diastolic,
        CASE
            WHEN b.systolic < 140 AND b.diastolic < 90 THEN 'Controlled'
            ELSE 'Uncontrolled'
        END AS bp_status
    FROM patients p
    JOIN most_recent_bp b ON p.patient_id = b.patient_id AND b.rn = 1
    WHERE p.patient_id IN (SELECT patient_id FROM (
        SELECT DISTINCT patient_id FROM conditions WHERE is_htn = true
    ))
      AND p.age BETWEEN 18 AND 85
      AND p.deceased = false
      AND p.patient_id NOT IN (
          SELECT DISTINCT patient_id FROM conditions WHERE is_esrd = true
      )
      AND p.patient_id NOT IN (
          SELECT DISTINCT patient_id FROM conditions WHERE is_hospice = true
      )
    ORDER BY bp_status, p.age
""").df()

controlled = len(numerator[numerator['bp_status'] == 'Controlled'])
uncontrolled = len(numerator[numerator['bp_status'] == 'Uncontrolled'])
rate = round(controlled / len(numerator) * 100, 1) if len(numerator) > 0 else 0

print("Denominator: ", len(numerator))
print("Controlled:  ", controlled)
print("Uncontrolled:", uncontrolled)
print("Measure Rate:", rate, "%")
print()
print(numerator[['name', 'age', 'gender', 'bp_date', 'systolic', 'diastolic', 'bp_status']].to_string(index=False))

Denominator:  46
Controlled:   31
Uncontrolled: 15
Measure Rate: 67.4 %

                               name  age gender    bp_date  systolic  diastolic    bp_status
         Leona665 Sau887 Nicolas769   55 female 2026-02-02       128         76   Controlled
      Irwin931 Claud279 Armstrong51   55   male 2025-05-12       102         58   Controlled
          Corinna386 Suk497 West559   55 female 2025-05-05       110         74   Controlled
      Dorian295 Nathan164 Rippin620   57   male 2025-06-18       105         75   Controlled
      Rubin812 Carter549 Collier206   60   male 2026-01-01       119         71   Controlled
       Willetta882 Oscar384 Rath779   60 female 2025-10-22       126         87   Controlled
            Creola518 Wintheiser220   60 female 2025-10-22       106         75   Controlled
     Cathi439 Shantell717 Ziemann98   60 female 2025-10-19        85         74   Controlled
         Danille883 Kari181 Conn188   61 female 2025-04-18       115         81   Controll

## Data Quality Check: Physiologic Plausibility of BP Readings

Before running measure logic, a clinical data pipeline requires physiologic plausibility checks. Raw EHR and synthetic data can contain values that pass structural validation but are biologically impossible in an ambulatory outpatient setting.

**Pulse pressure** (systolic minus diastolic) below 20 mmHg is incompatible with normal cardiac output in a living outpatient. Values in this range indicate a data entry error, device malfunction, or in the case of synthetic data, a simulator artifact.

A production pipeline would reject these readings before they enter the measure denominator. I apply that same standard here.

In [7]:
# Data quality audit: identify physiologically implausible BP readings
# Pulse pressure = systolic - diastolic
# Normal pulse pressure range is 40-60 mmHg
# Values below 20 mmHg are physiologically implausible in an ambulatory
# outpatient setting and suggest device error, data entry error,
# or in synthetic data, a simulator artifact

dq_check = con.execute("""
    SELECT
        patient_id,
        observation_date,
        systolic,
        diastolic,
        (systolic - diastolic) AS pulse_pressure
    FROM bp_observations
    WHERE (systolic - diastolic) < 20
    ORDER BY pulse_pressure
""").df()

total_bp = con.execute("SELECT COUNT(*) FROM bp_observations").fetchone()[0]

print("Total BP observations:          ", total_bp)
print("Implausible readings (PP < 20): ", len(dq_check))
print("Percent flagged:                ", round(len(dq_check) / total_bp * 100, 2), "%")
print()
if len(dq_check) > 0:
    print(dq_check.to_string(index=False))

Total BP observations:           2749
Implausible readings (PP < 20):  289
Percent flagged:                 10.51 %

                          patient_id observation_date  systolic  diastolic  pulse_pressure
db9b4fa8-bc4f-035e-6e9f-da4a0935e0a3       2016-03-21        88         92              -4
2fd76bb9-26ad-3cfc-8a7c-e1140d4417cb       2018-10-25        73         77              -4
0527fb92-a01c-a21f-3558-9f993ef1f66c       2022-07-02        76         80              -4
78bc7407-69f8-ac75-f142-f93d395a8638       2021-01-07        77         80              -3
78bc7407-69f8-ac75-f142-f93d395a8638       2023-06-15        83         85              -2
22c6264d-7cb6-a740-b51b-bea98b0c0bf5       2011-06-16       102        104              -2
27213d70-cb99-7dd9-7262-00cb9b21a347       1947-04-08        83         83               0
fdf35157-9b6a-08e9-8ced-9219342d8761       2020-02-09       100         98               2
fdf35157-9b6a-08e9-8ced-9219342d8761       2021-10-17        95 

## Data Quality Finding: Implausible BP Readings as a Clinical Workflow Signal

Running a pulse pressure audit on this dataset revealed that 10.5% of BP observations 
are physiologically implausible, including negative pulse pressures where diastolic 
exceeds systolic. In a living outpatient, this is impossible.

A pure data engineering response would be to filter and move on. A clinically informed approach 
asks the question: **why are these values in the system at all?**

Possible upstream causes:
- Staff entering systolic and diastolic values in reversed fields
- No real-time EHR validation alerting on impossible values before save
- Rushed rooming workflows creating transcription errors
- Device malfunction with no downstream quality check

**The two-pronged appropriate response:**
1. Apply a data quality filter for measure reporting purposes (values with pulse pressure < 10 mmHg are excluded as physiologically impossible in an ambulatory setting)
2. Escalate to clinical operations: Staff retraining, and an EHR-level input validation alert that fires before an impossible BP value can be saved

This is the difference between cleaning data and improving care. The measure rate is 
only as trustworthy as the readings behind it. Clinical expertise at the data pipeline 
level is what helps catch this. A data engineer filters it. A clinical informaticist flags 
it, explains it, and fixes it upstream.

In [8]:
# Apply data quality filter
# Exclude readings with pulse pressure < 10 mmHg (physiologically impossible)
# Readings in the 10-20 mmHg range are flagged above as a QI opportunity
# but retained here to avoid over-filtering synthetic data

before = con.execute("SELECT COUNT(*) FROM bp_observations").fetchone()[0]

con.execute("""
    DELETE FROM bp_observations
    WHERE (systolic - diastolic) < 10
""")

after = con.execute("SELECT COUNT(*) FROM bp_observations").fetchone()[0]

print("BP observations before filter:", before)
print("BP observations after filter: ", after)
print("Removed as physiologically impossible:", before - after)
print("Retained for QI review (PP 10-19):    ", 
      con.execute("""
          SELECT COUNT(*) FROM bp_observations 
          WHERE (systolic - diastolic) BETWEEN 10 AND 19
      """).fetchone()[0])

BP observations before filter: 2749
BP observations after filter:  2695
Removed as physiologically impossible: 54
Retained for QI review (PP 10-19):     235


In [9]:
# Final member-level audit table
# Combines denominator, numerator, DQ flags, and COVID proximity into one view

audit = con.execute("""
    WITH most_recent_bp AS (
        SELECT
            patient_id, systolic, diastolic, observation_date,
            ROW_NUMBER() OVER (PARTITION BY patient_id ORDER BY observation_date DESC) AS rn
        FROM bp_observations
    ),
    covid_dates AS (
        SELECT patient_id, onset_date AS covid_date
        FROM conditions WHERE is_covid = true
    ),
    covid_proximity AS (
        SELECT DISTINCT b.patient_id
        FROM bp_observations b
        JOIN covid_dates cd ON b.patient_id = cd.patient_id
        WHERE ABS(DATEDIFF('day', b.observation_date, cd.covid_date)) <= 365
    )
    SELECT
        p.name,
        p.age,
        p.gender,
        b.observation_date AS latest_bp_date,
        b.systolic,
        b.diastolic,
        CASE WHEN b.systolic < 140 AND b.diastolic < 90 THEN 'Controlled'
             ELSE 'Uncontrolled' END AS bp_status,
        CASE WHEN cp.patient_id IS NOT NULL THEN 'Yes' ELSE 'No' END AS covid_proximity_flag,
        CASE WHEN b.systolic < 140 AND b.diastolic < 90 THEN 'PASS'
             ELSE 'FAIL' END AS measure_result
    FROM patients p
    JOIN most_recent_bp b ON p.patient_id = b.patient_id AND b.rn = 1
    JOIN conditions c ON p.patient_id = c.patient_id AND c.is_htn = true
    LEFT JOIN covid_proximity cp ON p.patient_id = cp.patient_id
    WHERE p.age BETWEEN 18 AND 85
      AND p.deceased = false
      AND p.patient_id NOT IN (
          SELECT DISTINCT patient_id FROM conditions WHERE is_esrd = true
      )
      AND p.patient_id NOT IN (
          SELECT DISTINCT patient_id FROM conditions WHERE is_hospice = true
      )
    GROUP BY p.name, p.age, p.gender, b.observation_date,
             b.systolic, b.diastolic, cp.patient_id
    ORDER BY measure_result, covid_proximity_flag DESC, p.age
""").df()

total = len(audit)
passed = len(audit[audit['measure_result'] == 'PASS'])
failed = len(audit[audit['measure_result'] == 'FAIL'])
covid_flagged = len(audit[audit['covid_proximity_flag'] == 'Yes'])
rate = round(passed / total * 100, 1) if total > 0 else 0

print("HEDIS CBP Measure - Final Results")
print("="*50)
print("Denominator:         ", total)
print("Numerator (PASS):    ", passed)
print("Numerator (FAIL):    ", failed)
print("COVID flagged:       ", covid_flagged)
print("Measure Rate:        ", rate, "%")
print()
print(audit.to_string(index=False))

HEDIS CBP Measure - Final Results
Denominator:          46
Numerator (PASS):     31
Numerator (FAIL):     15
COVID flagged:        13
Measure Rate:         67.4 %

                               name  age gender latest_bp_date  systolic  diastolic    bp_status covid_proximity_flag measure_result
       Anton902 Bryan958 Strosin214   59   male     2025-07-19       139        104 Uncontrolled                  Yes           FAIL
     Elisha578 Jack927 Wilkinson796   59   male     2025-11-22       143         92 Uncontrolled                  Yes           FAIL
        Sherman440 Minh326 Pagac496   62   male     2025-10-26       125         98 Uncontrolled                  Yes           FAIL
      Tandra334 Pearlene638 Haag279   66 female     2026-03-04       120         94 Uncontrolled                  Yes           FAIL
         Nolan344 Kevin729 Lynch190   57   male     2025-04-22       107         90 Uncontrolled                   No           FAIL
            Song837 Tena12 Hauck852   